# 🍲 Nhận Diện Thức Ăn Bằng Trí Tuệ Nhân Tạo (Zero-Shot AI)

Phiên bản này sử dụng mô hình **CLIP của OpenAI**. Mô hình này **đã có sẵn trí thông minh**, hiểu được hình ảnh và văn bản. 
**ƯU ĐIỂM TUYỆT ĐỐI:** 
- **KHÔNG CẦN TRAIN (HỌC):** Chạy được luôn ngay lập tức!
- **KHÔNG CẦN DATASET:** Bạn thậm chí có thể xóa luôn thư mục `train`.
- Nhận diện siêu chuẩn dựa trên kiến thức khổng lồ của AI.

In [1]:
# 1. Cài đặt các thư viện AI mạnh nhất hiện nay (Chạy 1 lần)
!pip install transformers torch torchvision ipywidgets pillow opencv-python

  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl (12.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
facenet-pytorch 2.6.0 requires numpy<2.0.0,>=1.24.0, but you have numpy 2.4.6 which is incompatible.
facenet-pytorch 2.6.0 requires Pillow<10.3.0,>=10.2.0, but you have pillow 12.2.0 which is incompatible.
facenet-pytorch 2.6.0 requires torch<2.3.0,>=2.2.0, but you have torch 2.12.0+cpu which is incompatible.
facenet-pytorch 2.6.0 requires torchvision<0.18.0,>=0.17.0, but you have torchvision 0.27.0+cpu which is incompatible.
crewai 1.14.4 requires openai<3,>=2.30.0, but you have openai 2.24.0 which is incompatible.

[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import torch
from transformers import pipeline

print(" Đã tải xong thư viện!")

 Đã tải xong thư viện!


In [3]:
# 2. Thông tin các món ăn & Dịch sang tiếng Anh để AI hiểu
FOOD_INFO = {
    'Cá hú kho': {'english': 'Vietnamese braised fish in clay pot', 'price': '30,000 VNĐ', 'nutrition': '320 Kcal | 20g Protein | 22g Fat'},
    'Canh chua có cá': {'english': 'Vietnamese sour soup with fish', 'price': '25,000 VNĐ', 'nutrition': '180 Kcal | 15g Protein | 8g Fat'},
    'Canh chua không cá': {'english': 'Vietnamese sour soup with vegetables', 'price': '10,000 VNĐ', 'nutrition': '90 Kcal | 3g Protein | 2g Fat'},
    'Canh rau': {'english': 'clear vegetable soup', 'price': '8,000 VNĐ', 'nutrition': '45 Kcal | 2g Protein | 1g Fat'},
    'Cơm trắng': {'english': 'a bowl of white rice', 'price': '5,000 VNĐ', 'nutrition': '200 Kcal | 4g Protein | 45g Carbs'},
    'Đậu hũ sốt cà': {'english': 'fried tofu in tomato sauce', 'price': '15,000 VNĐ', 'nutrition': '150 Kcal | 10g Protein | 9g Fat'},
    'Rau xào': {'english': 'stir-fried green vegetables', 'price': '12,000 VNĐ', 'nutrition': '80 Kcal | 2g Protein | 6g Fat'},
    'Sườn nướng': {'english': 'grilled pork chops', 'price': '35,000 VNĐ', 'nutrition': '380 Kcal | 28g Protein | 24g Fat'},
    'Thịt kho': {'english': 'Vietnamese braised pork', 'price': '25,000 VNĐ', 'nutrition': '350 Kcal | 22g Protein | 25g Fat'},
    'Thịt kho trứng': {'english': 'Vietnamese braised pork with hard-boiled eggs', 'price': '30,000 VNĐ', 'nutrition': '400 Kcal | 25g Protein | 30g Fat'},
    'Trứng chiên': {'english': 'fried eggs', 'price': '10,000 VNĐ', 'nutrition': '120 Kcal | 10g Protein | 8g Fat'}
}

# Danh sách nhãn tiếng Anh để cho vào AI
candidate_labels = [info['english'] for info in FOOD_INFO.values()]

print(" Đã cấu hình Menu món ăn!")

 Đã cấu hình Menu món ăn!


In [4]:
# 3. Tải Siêu Mô Hình AI (Chỉ tải 1 lần, KHÔNG CẦN TRAIN)
print(" Đang khởi động AI OpenAI CLIP (Zero-shot)... (Có thể mất 1-2 phút lần đầu để tải)")
classifier = pipeline("zero-shot-image-classification", model="openai/clip-vit-large-patch14")
print(" SIÊU MÔ HÌNH ĐÃ SẴN SÀNG! Bạn có thể nhận diện ngay lập tức!")

 Đang khởi động AI OpenAI CLIP (Zero-shot)... (Có thể mất 1-2 phút lần đầu để tải)


model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

 SIÊU MÔ HÌNH ĐÃ SẴN SÀNG! Bạn có thể nhận diện ngay lập tức!


In [7]:
# 4. Giao Diện Nhận Diện
def predict_food(img_path):
    img = Image.open(img_path).convert('RGB')
    
    # Hiển thị ảnh
    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
    
    print("\n AI Đang suy nghĩ...")
    
    # Phân tích ảnh với AI
    result = classifier(img, candidate_labels=candidate_labels)
    
    best_match_english = result[0]['label']
    confidence = result[0]['score'] * 100
    
    # Tìm lại tên tiếng Việt tương ứng
    predicted_vietnamese = "Unknown"
    for vn_name, info in FOOD_INFO.items():
        if info['english'] == best_match_english:
            predicted_vietnamese = vn_name
            break
            
    info = FOOD_INFO.get(predicted_vietnamese, {})
    
    print("\n" + "="*50)
    print(f" NHẬN DIỆN MÓN ĂN: {predicted_vietnamese.upper()}")
    print(f"Độ tự tin của AI: {confidence:.2f}%")
    print("-"*50)
    if info:
        print(f" GIÁ TIỀN:    {info['price']}")
        print(f" DINH DƯỠNG:  {info['nutrition']}")
    print("="*50)

# Giao diện Upload
uploader = widgets.FileUpload(accept='image/*', multiple=False, description='Tải ảnh lên', button_style='success')
out = widgets.Output()

def on_upload(change):
    with out:
        clear_output()
        if not uploader.value:
            return
            
        if isinstance(uploader.value, dict):
            fname = list(uploader.value.keys())[0]
            content = uploader.value[fname]['content']
        else:
            content = uploader.value[0]['content']
            
        temp_path = 'temp_food_predict.jpg'
        with open(temp_path, 'wb') as f:
            f.write(content)
            
        predict_food(temp_path)
        uploader.value.clear() if hasattr(uploader.value, 'clear') else None

uploader.observe(on_upload, names='value')
display(widgets.VBox([
    widgets.HTML("<h3> BẤM NÚT ĐỂ TẢI ẢNH MÓN ĂN CỦA BẠN LÊN:</h3>"),
    uploader,
    out
]))